In [1]:
import os
import re
import csv
from collections import defaultdict
from statistics import mean, stdev

def parse_log_file_last_metrics(path):
    eval_metrics = {}
    with open(path, encoding="utf-8") as f:
        lines = f.readlines()
    # 倒序查找最后一个 start to eval
    i = len(lines) - 1
    while i >= 0:
        if lines[i].strip().startswith("start to eval"):
            # 期待接下来三行分别是 hit20、hit50、hit100
            if i + 1 < len(lines) and lines[i+1].startswith("hit20"):
                m = re.search(r"hit20\s+([0-9.eE+-]+)", lines[i+1])
                if m: eval_metrics["hit20"] = float(m.group(1))
            if i + 2 < len(lines) and lines[i+2].startswith("hit50"):
                m = re.search(r"hit50\s+([0-9.eE+-]+)", lines[i+2])
                if m: eval_metrics["hit50"] = float(m.group(1))
            if i + 3 < len(lines) and lines[i+3].startswith("hit100"):
                m = re.search(r"hit100\s+([0-9.eE+-]+)", lines[i+3])
                if m: eval_metrics["hit100"] = float(m.group(1))
            if i + 4 < len(lines) and lines[i+4].startswith("roc_auc"):
                fields = ['roc_auc', 'pr_auc', 'f1', 'mrr_pess', 'mrr_opt']
                nums = [float(x) for x in re.findall(r'[+-]?(?:\d+\.\d*|\.\d+|\d+)(?:[eE][+-]?\d+)?', lines[i+4])][-5:]
                result_dict = dict(zip(fields, nums))

                for k, v in result_dict.items():
                    eval_metrics[k] = v

            if eval_metrics:
                break
        i -= 1
    return eval_metrics

# ==== 用户指定参数 ====
# datasets = ["male"] # "twitter", "cora", "pubmed", "citeseer", "cora_ml", "icews18_max"]
datasets = ["ogbl_citation2"]
ratios = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
seeds = [1] # 之后记得改回 [1, 2, 3, 4, 5]
log_dir = "."

# ==== 处理每个数据集的日志文件 ====
for dataset in datasets:
    metric_bucket = defaultdict(list)
    metric_keys = set()
    
    for ratio in ratios:
        for seed in seeds:
            log_file = f"s{seed}-r{ratio}.log"
            log_path = os.path.join(log_dir, dataset, log_file)
            if not os.path.isfile(log_path):
                print(f"[WARN] 缺失: {log_path}")
                continue
            metrics = parse_log_file_last_metrics(log_path)
            if not metrics:
                print(f"[WARN] 无 test 指标: {log_path}")
                continue
            metric_bucket[ratio].append(metrics)
            metric_keys.update(metrics.keys())

    # ==== 结果输出 ====
    csv_path = f"./results/{dataset}_result.csv"
    metric_keys = sorted(metric_keys)
    header = ["split_ratio"] + [f"{k}" for k in metric_keys]

    with open(csv_path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=header)
        writer.writeheader()
        for ratio in sorted(metric_bucket):
            row = {"split_ratio": ratio}
            for k in metric_keys:
                vals = [m[k] for m in metric_bucket[ratio] if k in m]
                if vals:
                    mu = round(round(mean(vals), 6) * 100, 4)
                    sd = round(round(stdev(vals), 6) * 100 if len(vals) > 1 else 0.0, 4)
                    row[k] = f"{mu} ± {sd}"
                else:
                    row[k] = ""
            writer.writerow(row)
    print(f"[INFO] 写入完毕: {csv_path}")

[INFO] 写入完毕: ./results/ogbl_citation2_result.csv


In [4]:
import re

s1 = 'roc_auc, pr_auc, f1, mrr_pess, mrr_opt 0.6491436893907178 0.4360938378804934 0.39720289505101547 2.896194564527832e-05 0.08206523954868317\n'
s2 = 'roc_auc, pr_auc, f1, mrr 0.8886185413721194 0.711424252203574 0.7090104760892093 7.106179327820428e-05\n'

pat = r'[+-]?(?:\d+\.\d*|\.\d+|\d+)(?:[eE][+-]?\d+)?'

print(re.findall(pat, s1))
print(re.findall(pat, s2))

['1', '0.6491436893907178', '0.4360938378804934', '0.39720289505101547', '2.896194564527832e-05', '0.08206523954868317']
['1', '0.8886185413721194', '0.711424252203574', '0.7090104760892093', '7.106179327820428e-05']
